# Notebook consacré à l'analyse des variables nominales à choix multiples

Exemple :


Avez-vous des productions déposées sur Octaviana, la bibliothèque numérique de l'université ? (Plusieurs réponses possibles)
* Des travaux universitaires
* Des publications
* Des formations
* Des captations d'évènements (colloque, journée d'étude)
* Je ne connais pas Octaviana


In [ ]:
import pandas as pd

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-04-07.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df0

In [ ]:
list_nominal_multiple = [x for x in df_col.label.loc[(df_col.type=="nominal_multiple")&(df_col.label!="q12_1_how_humanum")]]
list_nominal_multiple

In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()

    return gb_data

In [ ]:
df0

# Types de données

In [ ]:
split_multiple_choices(df0, column='q33_sauvegarde_data', index = "q45_clé")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(8, figsize=(10, 15))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(list_nominal_multiple):
    gb_data = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total", y=col, data=gb_data,
                label="Total", color="b", ax=ax[n])
    sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])
    

sns.despine(left=True, bottom=True)
plt.savefig("multiple_choice.png")

In [ ]:
sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])